# Oxylabs Product Scraper

Scrapes product data (name, price, rating, stock, category) from the [Oxylabs sandbox](https://sandbox.oxylabs.io/products) across all paginated pages and persists results to both a **MySQL database** and a **CSV file**.

---
| Field | Value |
|---|---|
| Source | `https://sandbox.oxylabs.io/products` |
| Records | 3 000 (94 pages × 32 items) |
| Storage | MySQL · CSV |
| Driver | Selenium + ChromeDriver (headless) |

## 1 · Imports & Dependencies

In [1]:
from __future__ import annotations

import csv
import logging
import math
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import mysql.connector
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

# ── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

## 2 · Configuration

All tuneable parameters live in a single `Config` dataclass.  
Edit here — nowhere else.

In [2]:
@dataclass
class Config:
    # ── Scraping ──────────────────────────────────────────────────────────────
    base_url: str = "https://sandbox.oxylabs.io/products"
    total_results: int = 3_000
    items_per_page: int = 32
    page_delay_s: float = 1.0          # polite crawl delay between pages
    page_timeout_s: int = 10           # max seconds to wait for product cards

    # ── Output ────────────────────────────────────────────────────────────────
    csv_path: Path = Path("oxylabs_products.csv")

    # ── MySQL ─────────────────────────────────────────────────────────────────
    db_host: str = "localhost"
    db_user: str = "root"
    db_password: str = "lab2lab2"
    db_name: str = "oxylabs_db"

    # ── Derived (read-only) ───────────────────────────────────────────────────
    @property
    def total_pages(self) -> int:
        return math.ceil(self.total_results / self.items_per_page)


cfg = Config()
log.info("Config loaded — %d products across %d pages.", cfg.total_results, cfg.total_pages)

19:13:47  INFO      Config loaded — 3000 products across 94 pages.


## 3 · Database Setup

In [3]:
def get_connection(cfg: Config, include_db: bool = True) -> mysql.connector.MySQLConnection:
    """Return an open MySQL connection, optionally targeting the project database."""
    params = dict(host=cfg.db_host, user=cfg.db_user, password=cfg.db_password)
    if include_db:
        params["database"] = cfg.db_name
    return mysql.connector.connect(**params)


def setup_database(cfg: Config) -> None:
    """Create the database and *products* table if they do not already exist."""
    conn = get_connection(cfg, include_db=False)
    cursor = conn.cursor()
    try:
        cursor.execute(f"CREATE DATABASE IF NOT EXISTS `{cfg.db_name}`")
        cursor.execute(f"USE `{cfg.db_name}`")
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS products (
                id          INT AUTO_INCREMENT PRIMARY KEY,
                name        VARCHAR(255)  NOT NULL,
                price       VARCHAR(50),
                rating      VARCHAR(10),
                stock       VARCHAR(50),
                category    VARCHAR(255),
                scraped_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
        """)
        conn.commit()
        log.info("Database '%s' and table 'products' are ready.", cfg.db_name)
    finally:
        cursor.close()
        conn.close()


setup_database(cfg)

19:13:47  INFO      Database 'oxylabs_db' and table 'products' are ready.


## 4 · WebDriver Factory

In [4]:
def build_driver() -> webdriver.Chrome:
    """Construct a headless Chrome WebDriver with sensible defaults."""
    options = webdriver.ChromeOptions()
    for arg in ("--headless", "--no-sandbox", "--disable-dev-shm-usage"):
        options.add_argument(arg)
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

## 5 · Extraction Logic

> Each product card is parsed into a `Product` named-tuple for type safety and readability.

In [5]:
from typing import NamedTuple


class Product(NamedTuple):
    name: str
    price: str
    rating: str
    stock: str
    category: str


def extract_product(card) -> Optional[Product]:
    """
    Parse a single Selenium WebElement product card into a Product.

    Returns None if any critical field cannot be extracted.
    """
    try:
        name = card.find_element(By.CSS_SELECTOR, "h4.title").text.strip()

        category_parts = [
            span.text.strip()
            for span in card.find_elements(By.CSS_SELECTOR, "p.category span")
        ]
        category = ", ".join(category_parts)

        price = card.find_element(By.CSS_SELECTOR, "div.price-wrapper").text.strip()

        # Count filled (non-hollow) SVG stars
        star_svgs = card.find_elements(By.CSS_SELECTOR, "div.rating svg")
        filled = sum(
            1 for svg in star_svgs
            if "fill:currentColor" not in (svg.get_attribute("style") or "")
        )
        rating = f"{filled} / 5"

        try:
            stock_el = card.find_element(
                By.XPATH,
                ".//following-sibling::*[contains(text(),'In stock') "
                "or contains(text(),'Out of Stock')]",
            )
            stock = stock_el.text.strip()
        except NoSuchElementException:
            stock = "N/A"

        return Product(name=name, price=price, rating=rating,
                       stock=stock, category=category)

    except Exception as exc:  # noqa: BLE001
        log.warning("Could not parse product card: %s", exc)
        return None


def scrape_page(driver: webdriver.Chrome, page: int, cfg: Config) -> list[Product]:
    """Navigate to *page* and return all successfully parsed products."""
    driver.get(f"{cfg.base_url}?page={page}")
    try:
        WebDriverWait(driver, cfg.page_timeout_s).until(
            EC.presence_of_element_located((By.CLASS_NAME, "product-card"))
        )
    except TimeoutException:
        log.warning("Page %d timed out — skipping.", page)
        return []

    cards = driver.find_elements(By.CLASS_NAME, "product-card")
    log.info("Page %3d / %d — %d cards found.", page, cfg.total_pages, len(cards))

    products = [p for card in cards if (p := extract_product(card)) is not None]
    return products

## 6 · Main Scraping Loop

In [6]:
def run_scraper(cfg: Config) -> list[Product]:
    """Drive the full scrape and return all collected products."""
    all_products: list[Product] = []
    driver = build_driver()

    try:
        for page in range(1, cfg.total_pages + 1):
            page_products = scrape_page(driver, page, cfg)
            if not page_products:
                log.warning("No products returned on page %d — stopping early.", page)
                break
            all_products.extend(page_products)
            time.sleep(cfg.page_delay_s)
    finally:
        driver.quit()
        log.info("WebDriver closed.")

    log.info("Scraping complete — %d products collected.", len(all_products))
    return all_products


products = run_scraper(cfg)

19:13:47  INFO      ====== WebDriver manager ======
19:13:47  INFO      Get LATEST chromedriver version for google-chrome
19:13:48  INFO      Get LATEST chromedriver version for google-chrome
19:13:48  INFO      Driver [C:\Users\Admin\.wdm\drivers\chromedriver\win64\147.0.7727.117\chromedriver-win32/chromedriver.exe] found in cache
19:13:50  INFO      Page   1 / 94 — 32 cards found.
19:13:53  INFO      Page   2 / 94 — 32 cards found.
19:13:56  INFO      Page   3 / 94 — 32 cards found.
19:13:58  INFO      Page   4 / 94 — 32 cards found.
19:14:01  INFO      Page   5 / 94 — 32 cards found.
19:14:04  INFO      Page   6 / 94 — 32 cards found.
19:14:07  INFO      Page   7 / 94 — 32 cards found.
19:14:10  INFO      Page   8 / 94 — 32 cards found.
19:14:13  INFO      Page   9 / 94 — 32 cards found.
19:14:15  INFO      Page  10 / 94 — 32 cards found.
19:14:18  INFO      Page  11 / 94 — 32 cards found.
19:14:21  INFO      Page  12 / 94 — 32 cards found.
19:14:24  INFO      Page  13 / 94 — 32 car

## 7 · Persist to MySQL

In [7]:
def save_to_mysql(products: list[Product], cfg: Config) -> None:
    """Bulk-insert *products* into the MySQL products table."""
    if not products:
        log.warning("No products to insert — skipping MySQL save.")
        return

    conn = get_connection(cfg)
    cursor = conn.cursor()
    try:
        cursor.executemany(
            """
            INSERT INTO products (name, price, rating, stock, category)
            VALUES (%s, %s, %s, %s, %s)
            """,
            products,
        )
        conn.commit()
        log.info("%d rows inserted into MySQL.", cursor.rowcount)
    finally:
        cursor.close()
        conn.close()


save_to_mysql(products, cfg)

19:18:11  INFO      3000 rows inserted into MySQL.


## 8 · Export to CSV

In [8]:
def save_to_csv(products: list[Product], path: Path) -> None:
    """Write *products* to a UTF-8 CSV file at *path*."""
    if not products:
        log.warning("No products to export — skipping CSV save.")
        return

    with path.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        writer.writerow(Product._fields)   # header row from NamedTuple field names
        writer.writerows(products)

    log.info("CSV saved → %s (%d rows).", path.resolve(), len(products))


save_to_csv(products, cfg.csv_path)

19:18:11  INFO      CSV saved → D:\data eng\tech\lec\week 14\oxylabs_products.csv (3000 rows).


## 9 · Summary

In [9]:
import pandas as pd

df = pd.DataFrame(products, columns=Product._fields)

print(f"\n{'='*50}")
print(f"  Total products : {len(df):,}")
print(f"  Unique categories : {df['category'].nunique()}")
print(f"  In-stock rate : {(df['stock'] == 'In stock').mean():.1%}")
print(f"  Output CSV : {cfg.csv_path.resolve()}")
print(f"{'='*50}\n")

df.head(10)

19:18:12  INFO      Note: NumExpr detected 28 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
19:18:12  INFO      NumExpr defaulting to 16 threads.



  Total products : 3,000
  Unique categories : 704
  In-stock rate : 48.3%
  Output CSV : D:\data eng\tech\lec\week 14\oxylabs_products.csv



,name,price,rating,stock,category
0,The Legend of Zelda: Ocarina of Time,"91,99 €",5 / 5,N/A,"Action Adventure, Fantasy"
1,Super Mario Galaxy,"91,99 €",5 / 5,N/A,"Action, Platformer, 3D"
2,Super Mario Galaxy 2,"91,99 €",5 / 5,N/A,"Action, Platformer, 3D"
3,Metroid Prime,"89,99 €",5 / 5,N/A,"Action, Shooter, First-Person, Sci-Fi"
4,Super Mario Odyssey,"89,99 €",5 / 5,N/A,"Action, Platformer, 3D"
5,Halo: Combat Evolved,"87,99 €",5 / 5,N/A,"Action, Shooter, First-Person, Sci-Fi"
6,The House in Fata Morgana - Dreams of the Reve...,"83,99 €",5 / 5,N/A,"Adventure, Visual Novel"
7,NFL 2K1,"62,99 €",5 / 5,N/A,"Sports, Traditional, Football, Sim"
8,Uncharted 2: Among Thieves,"88,99 €",5 / 5,N/A,"Action Adventure, Modern, Linear"
9,Tekken 3,"91,99 €",5 / 5,N/A,"Action, Fighting, 3D"
